In [3]:
import requests
import pandas as pd
from datetime import datetime

# --- 1. 设置你的参数 ---

# 替换为你自己的 CoinGecko Demo API 密钥
# 你可以从这里免费注册: https://www.coingecko.com/en/api/pricing
API_KEY = 'CG-xtrHV9nWxQxtwq8tyThUtjNC' 

COIN_ID = 'ethereum'     # 你想查询的币种 (e.g., 'bitcoin', 'ethereum', 'aave')
VS_CURRENCY = 'usd'        # 对应的法币
DAYS = '3'                 # 请求的天数。根据API规则，2-90天会返回小时数据

# --- 2. 准备 API 请求 ---

base_url = "https://api.coingecko.com/api/v3"
endpoint = f"/coins/{COIN_ID}/market_chart"
url = f"{base_url}{endpoint}"

# 在 headers 中传递你的 API 密钥
headers = {
    'accept': 'application/json',
    'x-cg-demo-api-key': API_KEY
}

# 在 params 中传递查询参数
params = {
    'vs_currency': VS_CURRENCY,
    'days': DAYS
    # 我们不指定 'interval'，让 API 根据 'days=3' 自动选择小时粒度
}

# --- 3. 发送请求并处理数据 ---

print(f"正在向 CoinGecko 请求 {COIN_ID} 最近 {DAYS} 天的小时数据...")

try:
    # 发送 GET 请求
    response = requests.get(url, headers=headers, params=params)
    
    # 检查请求是否成功 (e.g., 200 OK)
    response.raise_for_status() 
    
    print("请求成功！正在处理数据...")
    data = response.json()
    
    # --- 4. 将数据转换为 Pandas DataFrame ---
    
    # 'prices' 键包含一个 [timestamp, price] 的列表
    prices_data = data['prices']
    
    # 创建 DataFrame
    df = pd.DataFrame(prices_data, columns=['timestamp', 'price'])
    
    # CoinGecko 返回的是毫秒级(ms)时间戳，我们需要将其转换为日期时间
    df['datetime'] = pd.to_datetime(df['timestamp'], unit='ms')
    
    # 将 datetime 设置为索引，并移除多余的 timestamp 列
    df = df.set_index('datetime')
    df = df.drop(columns=['timestamp'])
    
    # --- 5. 显示结果 ---
    
    print(f"\n成功获取 {COIN_ID} 的价格数据 ({len(df)} 个数据点):")
    print("\n--- 数据开头 ---")
    print(df.head())
    
    print("\n--- 数据结尾 ---")
    print(df.tail())
    
    # 验证数据粒度
    if len(df) > 0:
        # 计算两个连续数据点之间的时间差
        time_diff = df.index.to_series().diff().median()
        print(f"\n数据点之间的中位数间隔: {time_diff} (应接近1小时)")

except requests.exceptions.HTTPError as http_err:
    if response.status_code == 401:
        print(f"HTTP 错误 401: API 密钥无效或未提供。请检查 'API_KEY' 变量。")
    elif response.status_code == 429:
        print(f"HTTP 错误 429: 超出速率限制。请稍后再试。")
    else:
        print(f"HTTP 错误: {http_err}")
except requests.exceptions.RequestException as req_err:
    print(f"请求失败: {req_err}")
except Exception as e:
    print(f"处理数据时发生错误: {e}")

正在向 CoinGecko 请求 ethereum 最近 3 天的小时数据...
请求成功！正在处理数据...

成功获取 ethereum 的价格数据 (73 个数据点):

--- 数据开头 ---
                               price
datetime                            
2025-10-24 08:01:45.835  3944.832898
2025-10-24 09:01:57.402  3953.068296
2025-10-24 10:01:41.480  3957.118914
2025-10-24 11:02:02.851  3958.933731
2025-10-24 12:01:39.392  3948.359187

--- 数据结尾 ---
                               price
datetime                            
2025-10-27 04:01:05.618  4205.302812
2025-10-27 05:01:47.219  4216.255886
2025-10-27 06:01:27.287  4231.910340
2025-10-27 07:01:44.907  4232.019240
2025-10-27 07:13:38.000  4228.639352

数据点之间的中位数间隔: 0 days 00:59:55.447500 (应接近1小时)


In [4]:
import requests
import pandas as pd

# --- 1. 设置你的 API 密钥 ---

# 请确保这个变量仍然在内存中
# 或者从上一个单元格复制，并在这里重新粘贴你的 API 密钥
API_KEY = 'CG-xtrHV9nWxQxtwq8tyThUtjNC' 

# --- 2. 准备 API 请求 ---

url = "https://api.coingecko.com/api/v3/coins/list"
headers = {
    'accept': 'application/json',
    'x-cg-demo-api-key': API_KEY
}

# 这个端点不需要额外的参数
# params = {} 

print("正在向 CoinGecko 请求所有支持的币种列表...")
print("这个列表可能非常大，请稍候...")

try:
    # 发送 GET 请求
    response = requests.get(url, headers=headers)
    
    # 检查请求是否成功
    response.raise_for_status() 
    
    print("请求成功！正在将数据加载到 DataFrame...")
    data = response.json()
    
    # --- 3. 将数据转换为 Pandas DataFrame ---
    
    # data 是一个字典列表，例如:
    # [ {'id': 'bitcoin', 'symbol': 'btc', 'name': 'Bitcoin'}, ... ]
    df_coins = pd.DataFrame(data)
    
    # --- 4. 显示结果 ---
    
    print(f"\n成功获取！总共找到 {len(df_coins)} 个币种。")
    print("DataFrame 'df_coins' 已创建。")
    
    print("\n--- 币种列表 (前 10 行) ---")
    print(df_coins.head(10))
    
    print("\n--- 币种列表 (随机 5 行) ---")
    print(df_coins.sample(5))
    
    # --- 5. (重要) 如何使用这个列表 ---
    
    print("\n--- 如何查找你需要的 COIN_ID ---")
    print("你现在可以使用这个 'df_coins' DataFrame 来查找你想查询的币。")
    print("你需要的 'COIN_ID' 就是 'id' 这一列的值。\n")

    # 示例：如何查找 'Aave'
    try:
        search_symbol = 'aave'
        print(f"示例: 查找 symbol (小写) 为 '{search_symbol}' 的币")
        aave_result = df_coins[df_coins['symbol'] == search_symbol]
        print(aave_result)
        
        # 提取 'id'
        if not aave_result.empty:
            aave_id = aave_result.iloc[0]['id']
            print(f"-> Aave 的 COIN_ID 是: '{aave_id}' (这个值用于你上一个脚本)")
            
    except Exception as e:
        print(f"搜索示例出错: {e}")
        
    # 示例：如何查找 'Solana'
    try:
        search_name = 'Solana'
        print(f"\n示例: 查找 name 包含 '{search_name}' 的币 (忽略大小写)")
        solana_result = df_coins[df_coins['name'].str.contains(search_name, case=False, na=False)]
        print(solana_result)
    except Exception as e:
        print(f"搜索示例出错: {e}")
        

except requests.exceptions.HTTPError as http_err:
    if response.status_code == 401:
        print(f"HTTP 错误 401: API 密钥无效或未提供。请检查 'API_KEY' 变量。")
    elif response.status_code == 429:
        print(f"HTTP 错误 429: 超出速率限制。请稍后再试。")
    else:
        print(f"HTTP 错误: {http_err}")
except requests.exceptions.RequestException as req_err:
    print(f"请求失败: {req_err}")
except Exception as e:
    print(f"处理数据时发生错误: {e}")

正在向 CoinGecko 请求所有支持的币种列表...
这个列表可能非常大，请稍候...
请求成功！正在将数据加载到 DataFrame...

成功获取！总共找到 19345 个币种。
DataFrame 'df_coins' 已创建。

--- 币种列表 (前 10 行) ---
                                       id                            symbol  \
0                                       _                               gib   
1                             000-capital                               000   
2  01111010011110000110001001110100-token  01111010011110000110001001110100   
3                                  0chain                               zcn   
4                     0-knowledge-network                               0kn   
5                           0vix-protocol                               vix   
6                                      0x                               zrx   
7                0x0-ai-ai-smart-contract                               0x0   
8                     0x678-landwolf-1933                              wolf   
9                             0xgasless-2                         

In [5]:
df_coins

,id,symbol,name
0,_,gib,༼ つ ◕_◕ ༽つ
1,000-capital,000,000 Capital
2,01111010011110000110001001110100-token,01111010011110000110001001110100,01111010011110000110001001110100
3,0chain,zcn,Zus
4,0-knowledge-network,0kn,0 Knowledge Network
...,...,...,...
19340,zygo-the-frog-2,zygo,Zygo The Frog
19341,zyncoin-2,zyn,ZynCoin
19342,zynecoin,zyn,Zynecoin
19343,zyro-2,zyro,ZYRO


In [6]:
df_coins.to_csv('coingecko_supported_coins.csv', index=False)